# DhVaani Android — Colab Build

This notebook builds `app-debug.apk` ( fully offline Hindi voice cloner) from source.

**What it does**
1. Installs JDK 17 + Android SDK 34 + Gradle 8.7.
2. Downloads the ONNX graphs and feature assets (with retry + size check).
3. Converts the `.npz` features to the app's `.bin` format.
4. Runs `assembleDebug`.

**Critical:** `JAVA_HOME` must point to Java 17 and be **exported to the Gradle subprocess** — AGP 8.5 requires Java 17 (Colab defaults to Java 11, so the build would otherwise fail).

In [ ]:
!set -e
# ---- JDK 17 ----
!apt-get -y -qq install openjdk-17-jdk-headless >/dev/null 2>&1 || true
!java -version
!export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64 && $JAVA_HOME/bin/javac -version

# ---- Android SDK 34 (cmdline-tools) ----
import os, urllib.request, zipfile
os.makedirs('/opt/android-sdk/cmdline-tools', exist_ok=True)
url = 'https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip'
print('downloading cmdline-tools ...')
urllib.request.urlretrieve(url, '/tmp/clt.zip')
with zipfile.ZipFile('/tmp/clt.zip') as z: z.extractall('/tmp/clt')
os.rename('/tmp/clt/cmdline-tools', '/opt/android-sdk/cmdline-tools/latest')

os.environ['ANDROID_HOME'] = '/opt/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/opt/android-sdk'
os.environ['PATH'] = '/opt/android-sdk/cmdline-tools/latest/bin:' + os.environ['PATH']

!yes | /opt/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses >/dev/null 2>&1 || true
!/opt/android-sdk/cmdline-tools/latest/bin/sdkmanager --install "platform-tools" "platforms;android-34" "build-tools;34.0.0" 2>&1 | tail -5

# ---- Gradle 8.7 ----
!curl -sSL -o /tmp/gradle.zip https://services.gradle.org/distributions/gradle-8.7-bin.zip
!unzip -q /tmp/gradle.zip -d /opt
print('tools ready')

In [ ]:
import os, subprocess, sys, time

# Build environment — MUST be Java 17 and exported to the subprocess.
env = dict(os.environ)
env['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
env['ANDROID_HOME'] = '/opt/android-sdk'
env['ANDROID_SDK_ROOT'] = '/opt/android-sdk'
env['PATH'] = ('/opt/android-sdk/platform-tools:'
              '/opt/android-sdk/cmdline-tools/latest/bin:'
              '/opt/gradle-8.7/bin:' + env['PATH'])

def run(cmd, **kw):
    print('$', cmd)
    subprocess.check_call(cmd, shell=True, env=env, **kw)

run('java -version && gradle --version | head -5')

# ---- Download models with retry + min-byte-size check ----
import urllib.request
HF = 'https://huggingface.co/Bbkblo/DhVaani-0.5-ONNX/resolve/main'
ASSETS = 'app/src/main/assets'
os.makedirs(ASSETS, exist_ok=True)
GOALS = [
    ('text_encoder_int8.onnx', 5_000_000),
    ('fm_decoder_int8.onnx',   110_000_000),
    ('vocoder_backbone.onnx',   30_000_000),
    ('mel_fb.npz',              200_000),
    ('vocos_head.npz',          1_000_000),
    ('tokens.txt',                1_000),
]
for name, minbytes in GOALS:
    target = os.path.join(ASSETS, name)
    for attempt in range(1, 6):
        try:
            print(f'downloading {name} (attempt {attempt}) ...')
            urllib.request.urlretrieve(f'{HF}/{name}', target)
            size = os.path.getsize(target)
            if size >= minbytes:
                print(f'  OK {name} {size} bytes')
                break
            print(f'  too small ({size}) retrying')
            os.remove(target)
        except Exception as e:
            print(f'  error {e} retrying')
            time.sleep(2)
    else:
        raise SystemExit(f'Failed to download {name}')

# ---- Convert npz -> bin ----
run('python3 -m pip -q install numpy')
run(f'python3 scripts/make_assets.py --in {ASSETS} --out {ASSETS}')

# ---- Assemble ----
run('gradle assembleDebug --stacktrace')
print('\nBUILD COMPLETE')
print('APK: ' + os.path.abspath('app/build/outputs/apk/debug/app-debug.apk'))

Download the APK from `app/build/outputs/apk/debug/app-debug.apk` (in the Colab file browser, or `files.download(...)`).

**Notes**
* The int8 models are ~180 MB total, so the APK is ~250 MB. For Play publishing use an App Bundle + asset packs or download models on first launch.
* If the fm_decoder download truncates, the retry-loop re-downloads until the size check passes.